# Recorte de rasters por humedal

Este notebook documenta el proceso de separación espacial del raster multibanda por humedal.

A partir de un stack Sentinel-2 y una capa vectorial de humedales, se generan:

- raster recortado por humedal;
- máscara binaria por humedal;
- tabla resumen de productos generados.

Estos productos funcionan como insumos intermedios para las etapas posteriores del flujo de clasificación de coberturas.

In [1]:
from pathlib import Path
import sys

In [2]:
BASE_DIR = Path.cwd()

while not (BASE_DIR / "src").exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent

sys.path.append(str(BASE_DIR))

print("BASE_DIR detectado:")
print(BASE_DIR)

BASE_DIR detectado:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml


## Importación de funciones auxiliares

La lógica principal para recortar rasters por humedal se encuentra en:

`src/clasificacion_coberturas/procesamiento_raster.py`

Este módulo permite cargar la capa vectorial de humedales, validar el campo de identificación, ajustar CRS, recortar el raster, generar máscaras binarias y guardar un resumen de salidas.

In [3]:
from src.clasificacion_coberturas.procesamiento_raster import procesar_recortes_humedales

## Configuración de rutas

Se definen las rutas del raster multibanda, la capa de humedales y las carpetas de salida.

Las carpetas `RPR_<año>` y `MBR_<año>` se conservan como nombres de trabajo:

- `RPR`: raster por región o raster recortado por humedal;
- `MBR`: máscara binaria raster por humedal.

In [4]:
# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

year = 2018

ruta_raster = (
    BASE_DIR
    / "data"
    / "raw"
    / "clasificacion_coberturas"
    / "rasters"
    / "S2_2018_STACK_3.tif"
)

ruta_humedales = (
    BASE_DIR
    / "data"
    / "raw"
    / "clasificacion_coberturas"
    / "vectores"
    / "ROI_HUMEDALES.shp"
)

carpeta_rasters_recortados = (
    BASE_DIR
    / "data"
    / "interim"
    / "clasificacion_coberturas"
    / f"RPR_{year}"  # RPR: raster recortado por humedal
)

carpeta_mascaras = (
    BASE_DIR
    / "data"
    / "interim"
    / "clasificacion_coberturas"
    / f"MBR_{year}"  # MBR: máscara binaria raster por humedal
)

carpeta_resumen = (
    BASE_DIR
    / "outputs"
    / "tables"
    / "clasificacion_coberturas"
)

print("Raster de entrada:")
print(ruta_raster)

print("\nCapa de humedales:")
print(ruta_humedales)

print("\nCarpeta de raster recortados:")
print(carpeta_rasters_recortados)

print("\nCarpeta de máscaras:")
print(carpeta_mascaras)

print("\nCarpeta de resumen:")
print(carpeta_resumen)

Raster de entrada:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\data\raw\clasificacion_coberturas\rasters\S2_2018_STACK_3.tif

Capa de humedales:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\data\raw\clasificacion_coberturas\vectores\ROI_HUMEDALES.shp

Carpeta de raster recortados:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\data\interim\clasificacion_coberturas\RPR_2018

Carpeta de máscaras:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\data\interim\clasificacion_coberturas\MBR_2018

Carpeta de resumen:
C:\Users\AVDON\JupyterLab\ATENEA\humedales_bogota_ml\outputs\tables\clasificacion_coberturas


## Configuración de parámetros

El campo `nombre_ap` identifica cada humedal dentro de la capa vectorial. Este campo se usa para disolver geometrías y nombrar los archivos de salida.

El parámetro `all_touched` controla cómo se incluyen los píxeles en el recorte y la máscara:

- `False`: incluye principalmente píxeles cuyo centro cae dentro del polígono;
- `True`: incluye todos los píxeles tocados por el borde del polígono.

Usar `True` puede ser útil cuando se busca una inclusión más amplia en bordes o polígonos estrechos, aunque puede incorporar más píxeles periféricos.

In [5]:
campo_nombre = "nombre_ap"
nodata_val = -9999
all_touched = False

print("Campo de nombre:", campo_nombre)
print("Valor NoData:", nodata_val)
print("all_touched:", all_touched)

Campo de nombre: nombre_ap
Valor NoData: -9999
all_touched: False


## Ejecución del recorte por humedal

Se ejecuta el procesamiento completo:

1. carga del raster multibanda;
2. carga y validación de la capa de humedales;
3. reproyección de humedales al CRS del raster, si es necesario;
4. disolución por nombre de humedal;
5. recorte del raster por humedal;
6. generación de máscara binaria;
7. exportación de productos;
8. generación de tabla resumen.

In [6]:
df_resumen_recortes = procesar_recortes_humedales(
    ruta_raster=ruta_raster,
    ruta_humedales=ruta_humedales,
    carpeta_rasters_recortados=carpeta_rasters_recortados,
    carpeta_mascaras=carpeta_mascaras,
    carpeta_resumen=carpeta_resumen,
    year=year,
    campo_nombre=campo_nombre,
    nodata_val=nodata_val,
    all_touched=all_touched
)

df_resumen_recortes

Reproyectando humedales de GEOGCS["unknown",DATUM["D_Unknown_based_on_GRS_1980_ellipsoid",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AXIS["Longitude",EAST],AXIS["Latitude",NORTH]] a EPSG:4326...
Procesado: PDEM Entre Nubes-Cerro Juan Rey
Procesado: PDEM Entre Nubes-Cuchilla Guacamayas
Procesado: PDEM Entre Nubes-Cuchilla el Gavilán
Procesado: PDEM Mirador de los Nevados
Procesado: PDEM Serranía del Zuque
Procesado: PDEM Soratama
Procesado: RDH Chiguasuque ? La Isla
Procesado: RDH Complejo de Humedales El Tunjo
Procesado: RDH Salitre
Procesado: RDH Tingua Azul
Procesado: RDH de Capellanía o La Cofradía
Procesado: RDH de Córdoba  Niza
Procesado: RDH de Jaboque
Procesado: RDH de Juan Amarillo o Tibabuyes
Procesado: RDH de La Conejera
Procesado: RDH de Santa María del Lago
Procesado: RDH de Techo
Procesado: RDH de Tibanica
Procesado: RDH de Torca y Guaymaral
Procesado: RDH de la Vaca
Procesado: RDH del Burro


,year,nombre_humedal,slug,ruta_raster_recortado,ruta_mascara,all_touched,nodata_val
0,2018,PDEM Entre Nubes-Cerro Juan Rey,pdem_entre_nubes_cerro_juan_rey,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
1,2018,PDEM Entre Nubes-Cuchilla Guacamayas,pdem_entre_nubes_cuchilla_guacamayas,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
2,2018,PDEM Entre Nubes-Cuchilla el Gavilán,pdem_entre_nubes_cuchilla_el_gavilan,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
3,2018,PDEM Mirador de los Nevados,pdem_mirador_de_los_nevados,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
4,2018,PDEM Serranía del Zuque,pdem_serrania_del_zuque,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
5,2018,PDEM Soratama,pdem_soratama,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
6,2018,RDH Chiguasuque ? La Isla,rdh_chiguasuque_la_isla,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
7,2018,RDH Complejo de Humedales El Tunjo,rdh_complejo_de_humedales_el_tunjo,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
8,2018,RDH Salitre,rdh_salitre,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999
9,2018,RDH Tingua Azul,rdh_tingua_azul,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,C:\Users\AVDON\JupyterLab\ATENEA\humedales_bog...,False,-9999


## Producto de esta etapa

Al finalizar este notebook, se generan raster recortados y máscaras binarias para cada humedal.

Los raster recortados se guardan en:

`data/interim/clasificacion_coberturas/RPR_<año>/`

Las máscaras binarias se guardan en:

`data/interim/clasificacion_coberturas/MBR_<año>/`

Además, se genera una tabla resumen con fecha de creación en:

`outputs/tables/clasificacion_coberturas/`

## Siguiente etapa del flujo

Una vez generados los raster recortados por humedal, el siguiente paso consiste en preparar el raster que será usado para análisis de variables, selección de bandas e insumos del modelo de clasificación.

Esta fase se documenta en:

`03_preparacion_raster.ipynb`